# Machine Learning with MLlib

# Designing Machine Learning Pipelines

## Data Ingestion and Exploration

In [1]:
from pyspark.sql import SparkSession

spark = (SparkSession
  .builder
  .appName("MLlibExampleApp")
  .getOrCreate())

filePath = """../data/learning-spark-v2/sf-airbnb/sf-airbnb-clean.parquet/"""
airbnbDF = spark.read.parquet(filePath)
airbnbDF.select("neighbourhood_cleansed", "room_type", "bedrooms", "bathrooms", "number_of_reviews", "price").show(5)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/22 03:57:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+----------------------+---------------+--------+---------+-----------------+-----+
|neighbourhood_cleansed|      room_type|bedrooms|bathrooms|number_of_reviews|price|
+----------------------+---------------+--------+---------+-----------------+-----+
|      Western Addition|Entire home/apt|     1.0|      1.0|            180.0|170.0|
|        Bernal Heights|Entire home/apt|     2.0|      1.0|            111.0|235.0|
|        Haight Ashbury|   Private room|     1.0|      4.0|             17.0| 65.0|
|        Haight Ashbury|   Private room|     1.0|      4.0|              8.0| 65.0|
|      Western Addition|Entire home/apt|     2.0|      1.5|             27.0|785.0|
+----------------------+---------------+--------+---------+-----------------+-----+
only showing top 5 rows



## Creating Training and Test Data Sets

In [2]:
trainDF, testDF = airbnbDF.randomSplit([.8, .2], seed=42)
print(f"""There are {trainDF.count()} rows in the training set, and {testDF.count()} in the test set""")

26/04/22 03:57:28 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


There are 5780 rows in the training set, and 1366 in the test set


## Preparing Features with Transformers

In [3]:
from pyspark.ml.feature import VectorAssembler

vecAssembler = VectorAssembler(inputCols=["bedrooms"], outputCol="features")
vecTrainDF = vecAssembler.transform(trainDF)
vecTrainDF.select("bedrooms", "features", "price").show(10)

+--------+--------+-----+
|bedrooms|features|price|
+--------+--------+-----+
|     1.0|   [1.0]|200.0|
|     1.0|   [1.0]|130.0|
|     1.0|   [1.0]| 95.0|
|     1.0|   [1.0]|250.0|
|     3.0|   [3.0]|250.0|
|     1.0|   [1.0]|115.0|
|     1.0|   [1.0]|105.0|
|     1.0|   [1.0]| 86.0|
|     1.0|   [1.0]|100.0|
|     2.0|   [2.0]|220.0|
+--------+--------+-----+
only showing top 10 rows



## Understanding Linear Regression

## Using Estimators to Build Models

In [4]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(featuresCol="features", labelCol="price")
lrModel = lr.fit(vecTrainDF)

m = round(lrModel.coefficients[0], 2)
b = round(lrModel.intercept, 2)
print(f"""The formula for the linear regression line is price = {m}*bedrooms + {b}""")

26/04/22 03:57:31 WARN Instrumentation: [7612d9cc] regParam is zero, which might cause numerical instability and overfitting.
26/04/22 03:57:32 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/04/22 03:57:32 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


The formula for the linear regression line is price = 123.68*bedrooms + 47.51


## Creating a Pipeline

In [5]:
from pyspark.ml import Pipeline

pipeline = Pipeline(stages=[vecAssembler, lr])
pipelineModel = pipeline.fit(trainDF)

predDF = pipelineModel.transform(testDF)
predDF.select("bedrooms", "features", "price", "prediction").show(10)

26/04/22 03:57:33 WARN Instrumentation: [26e3e73e] regParam is zero, which might cause numerical instability and overfitting.


+--------+--------+------+------------------+
|bedrooms|features| price|        prediction|
+--------+--------+------+------------------+
|     1.0|   [1.0]|  85.0|171.18598011578285|
|     1.0|   [1.0]|  45.0|171.18598011578285|
|     1.0|   [1.0]|  70.0|171.18598011578285|
|     1.0|   [1.0]| 128.0|171.18598011578285|
|     1.0|   [1.0]| 159.0|171.18598011578285|
|     2.0|   [2.0]| 250.0|294.86172649777757|
|     1.0|   [1.0]|  99.0|171.18598011578285|
|     1.0|   [1.0]|  95.0|171.18598011578285|
|     1.0|   [1.0]| 100.0|171.18598011578285|
|     1.0|   [1.0]|2010.0|171.18598011578285|
+--------+--------+------+------------------+
only showing top 10 rows



In [6]:
# One-hot encoding
from pyspark.ml.feature import OneHotEncoder, StringIndexer

categoricalCols = [field for (field, dataType) in trainDF.dtypes if dataType == "string"]
indexOutputCols = [x + "Index" for x in categoricalCols]
oheOutputCols = [x + "OHE" for x in categoricalCols]

stringIndexer = StringIndexer(inputCols=categoricalCols,
  outputCols=indexOutputCols,
  handleInvalid="skip")
oheEncoder = OneHotEncoder(inputCols=indexOutputCols,
  outputCols=oheOutputCols)

numericCols = [field for (field, dataType) in trainDF.dtypes if ((dataType == "double") & (field != "price"))]

assemblerInputs = oheOutputCols + numericCols
vecAssembler = VectorAssembler(inputCols=assemblerInputs, outputCol="features")

# rest
lr = LinearRegression(labelCol="price", featuresCol="features")
pipeline = Pipeline(stages = [stringIndexer, oheEncoder, vecAssembler, lr])
pipelineModel = pipeline.fit(trainDF)
predDF = pipelineModel.transform(testDF)
predDF.select("features", "price", "prediction").show(5)

26/04/22 03:57:35 WARN Instrumentation: [a7f6ee1d] regParam is zero, which might cause numerical instability and overfitting.


+--------------------+-----+------------------+
|            features|price|        prediction|
+--------------------+-----+------------------+
|(98,[0,3,6,22,43,...| 85.0| 55.24365707389188|
|(98,[0,3,6,22,43,...| 45.0|23.357685914717877|
|(98,[0,3,6,22,43,...| 70.0|28.474464479034395|
|(98,[0,3,6,12,42,...|128.0| -91.6079079594947|
|(98,[0,3,6,12,43,...|159.0| 95.05688229945372|
+--------------------+-----+------------------+
only showing top 5 rows



## Evaluating Models

In [7]:
from pyspark.ml.evaluation import RegressionEvaluator

regressionEvaluator = RegressionEvaluator(
  predictionCol="prediction",
  labelCol="price",
  metricName="rmse")
rmse = regressionEvaluator.evaluate(predDF)
print(f"RMSE is {rmse:.1f}")

RMSE is 220.6


In [8]:
r2 = regressionEvaluator.setMetricName("r2").evaluate(predDF)
print(f"R2 is {r2}")

R2 is 0.16043316698848087


## Saving and Loading Models

In [9]:
pipelinePath = "/tmp/lr-pipeline-model"
pipelineModel.write().overwrite().save(pipelinePath)

In [10]:
from pyspark.ml import PipelineModel

savedPipelineModel = PipelineModel.load(pipelinePath)
# use loaded model
predDF = pipelineModel.transform(testDF)
predDF.select("features", "price", "prediction").show(5)

+--------------------+-----+------------------+
|            features|price|        prediction|
+--------------------+-----+------------------+
|(98,[0,3,6,22,43,...| 85.0| 55.24365707389188|
|(98,[0,3,6,22,43,...| 45.0|23.357685914717877|
|(98,[0,3,6,22,43,...| 70.0|28.474464479034395|
|(98,[0,3,6,12,42,...|128.0| -91.6079079594947|
|(98,[0,3,6,12,43,...|159.0| 95.05688229945372|
+--------------------+-----+------------------+
only showing top 5 rows



# Hyperparameter Tuning

## Tree-Based Models

### Decision trees

In [11]:
from pyspark.ml.regression import DecisionTreeRegressor

dt = DecisionTreeRegressor(labelCol="price")

# Filter for just numeric columns (and exclude price, our label)
numericCols = [field for (field, dataType) in trainDF.dtypes if ((dataType == "double") & (field != "price"))]

# Combine output of StringIndexer defined above and numeric columns
assemblerInputs = indexOutputCols + numericCols
vecAssembler = VectorAssembler(inputCols=assemblerInputs, outputCol="features")

# Combine stages into pipeline
stages = [stringIndexer, vecAssembler, dt]
pipeline = Pipeline(stages=stages)
dt.setMaxBins(40) # 
pipelineModel = pipeline.fit(trainDF)

In [12]:
dtModel = pipelineModel.stages[-1]
# print(dtModel.toDebugString)

import pandas as pd
featureImp = pd.DataFrame(
  list(zip(vecAssembler.getInputCols(), dtModel.featureImportances)),
  columns=["feature", "importance"])
featureImp.sort_values(by="importance", ascending=False)[:10]

,feature,importance
12,bedrooms,0.283406
1,cancellation_policyIndex,0.167893
2,instant_bookableIndex,0.140081
4,property_typeIndex,0.128179
15,number_of_reviews,0.126233
3,neighbourhood_cleansedIndex,0.056200
9,longitude,0.038810
14,minimum_nights,0.029473
13,beds,0.015218
5,room_typeIndex,0.010905


In [13]:
predDF = pipelineModel.transform(testDF)
predDF.select("features", "price", "prediction").show(5)

+--------------------+-----+------------------+
|            features|price|        prediction|
+--------------------+-----+------------------+
|(33,[1,3,4,7,8,9,...| 85.0|131.96658097686375|
|[0.0,2.0,0.0,15.0...| 45.0|104.23992784125075|
|[0.0,2.0,0.0,15.0...| 70.0|104.23992784125075|
|(33,[1,3,5,7,8,9,...|128.0|104.23992784125075|
|(33,[1,3,4,5,7,8,...|159.0|104.23992784125075|
+--------------------+-----+------------------+
only showing top 5 rows



### Random forests

In [14]:
from pyspark.ml.regression import RandomForestRegressor
rf = RandomForestRegressor(labelCol="price", maxBins=40, seed=42)

numericCols = [field for (field, dataType) in trainDF.dtypes if ((dataType == "double") & (field != "price"))]

# Combine output of StringIndexer defined above and numeric columns
assemblerInputs = indexOutputCols + numericCols
vecAssembler = VectorAssembler(inputCols=assemblerInputs, outputCol="features")

# Combine stages into pipeline
stages = [stringIndexer, vecAssembler, rf]
pipeline = Pipeline(stages=stages)
pipelineModel = pipeline.fit(trainDF)

In [15]:
rfModel = pipelineModel.stages[-1]
# print(rfModel.toDebugString)

import pandas as pd
featureImp = pd.DataFrame(
  list(zip(vecAssembler.getInputCols(), rfModel.featureImportances)),
  columns=["feature", "importance"])
featureImp.sort_values(by="importance", ascending=False)[:10]

,feature,importance
10,accommodates,0.141341
1,cancellation_policyIndex,0.128434
12,bedrooms,0.123870
3,neighbourhood_cleansedIndex,0.111406
8,latitude,0.097640
13,beds,0.081530
4,property_typeIndex,0.074584
11,bathrooms,0.060642
14,minimum_nights,0.045775
5,room_typeIndex,0.026790


In [16]:
predDF = pipelineModel.transform(testDF)
predDF.select("features", "price", "prediction").show(5)

+--------------------+-----+------------------+
|            features|price|        prediction|
+--------------------+-----+------------------+
|(33,[1,3,4,7,8,9,...| 85.0| 136.9326183156069|
|[0.0,2.0,0.0,15.0...| 45.0|  89.8282663067711|
|[0.0,2.0,0.0,15.0...| 70.0| 92.05748540632366|
|(33,[1,3,5,7,8,9,...|128.0|100.27172015928167|
|(33,[1,3,4,5,7,8,...|159.0|111.24683151062177|
+--------------------+-----+------------------+
only showing top 5 rows



## k-Fold Cross-Validation

In [17]:
pipeline = Pipeline(stages = [stringIndexer, vecAssembler, rf]) # rf

from pyspark.ml.tuning import ParamGridBuilder
from pyspark.ml.tuning import CrossValidator

paramGrid = (ParamGridBuilder()
  .addGrid(rf.maxDepth, [2, 4, 6])
  .addGrid(rf.numTrees, [10, 100])
  .build())

evaluator = RegressionEvaluator(labelCol="price",
  predictionCol="prediction",
  metricName="rmse")

cv = CrossValidator(estimator=pipeline,
  evaluator=evaluator,
  estimatorParamMaps=paramGrid,
  numFolds=3,
  seed=42)
cvModel = cv.fit(trainDF)

list(zip(cvModel.getEstimatorParamMaps(), cvModel.avgMetrics))

26/04/22 03:57:56 WARN DAGScheduler: Broadcasting large task binary with size 1324.6 KiB
26/04/22 03:58:04 WARN DAGScheduler: Broadcasting large task binary with size 1158.1 KiB
26/04/22 03:58:12 WARN DAGScheduler: Broadcasting large task binary with size 1199.0 KiB
26/04/22 03:58:15 WARN DAGScheduler: Broadcasting large task binary with size 1225.2 KiB


[({Param(parent='RandomForestRegressor_ea56a97ce9bf', name='maxDepth', doc='Maximum depth of the tree. (>= 0) E.g., depth 0 means 1 leaf node; depth 1 means 1 internal node + 2 leaf nodes. Must be in range [0, 30].'): 2,
   Param(parent='RandomForestRegressor_ea56a97ce9bf', name='numTrees', doc='Number of trees to train (>= 1).'): 10},
  np.float64(291.1822640924783)),
 ({Param(parent='RandomForestRegressor_ea56a97ce9bf', name='maxDepth', doc='Maximum depth of the tree. (>= 0) E.g., depth 0 means 1 leaf node; depth 1 means 1 internal node + 2 leaf nodes. Must be in range [0, 30].'): 2,
   Param(parent='RandomForestRegressor_ea56a97ce9bf', name='numTrees', doc='Number of trees to train (>= 1).'): 100},
  np.float64(286.7714750274078)),
 ({Param(parent='RandomForestRegressor_ea56a97ce9bf', name='maxDepth', doc='Maximum depth of the tree. (>= 0) E.g., depth 0 means 1 leaf node; depth 1 means 1 internal node + 2 leaf nodes. Must be in range [0, 30].'): 4,
   Param(parent='RandomForestRegre

In [18]:
predDF = cvModel.transform(testDF)
predDF.select("features", "price", "prediction").show(5)

+--------------------+-----+------------------+
|            features|price|        prediction|
+--------------------+-----+------------------+
|(33,[1,3,4,7,8,9,...| 85.0| 143.8612635543596|
|[0.0,2.0,0.0,15.0...| 45.0| 90.56831409179672|
|[0.0,2.0,0.0,15.0...| 70.0|  91.0161235570984|
|(33,[1,3,5,7,8,9,...|128.0| 93.10621215807369|
|(33,[1,3,4,5,7,8,...|159.0|113.12925185885474|
+--------------------+-----+------------------+
only showing top 5 rows



## Optimizing Pipelines

In [19]:
cvModel = cv.setParallelism(4).fit(trainDF)

list(zip(cvModel.getEstimatorParamMaps(), cvModel.avgMetrics))

26/04/22 03:58:19 WARN DAGScheduler: Broadcasting large task binary with size 1324.6 KiB
26/04/22 03:58:23 WARN DAGScheduler: Broadcasting large task binary with size 1158.1 KiB
26/04/22 03:58:27 WARN DAGScheduler: Broadcasting large task binary with size 1199.0 KiB
26/04/22 03:58:30 WARN DAGScheduler: Broadcasting large task binary with size 1225.2 KiB


[({Param(parent='RandomForestRegressor_ea56a97ce9bf', name='maxDepth', doc='Maximum depth of the tree. (>= 0) E.g., depth 0 means 1 leaf node; depth 1 means 1 internal node + 2 leaf nodes. Must be in range [0, 30].'): 2,
   Param(parent='RandomForestRegressor_ea56a97ce9bf', name='numTrees', doc='Number of trees to train (>= 1).'): 10},
  np.float64(291.1822640924783)),
 ({Param(parent='RandomForestRegressor_ea56a97ce9bf', name='maxDepth', doc='Maximum depth of the tree. (>= 0) E.g., depth 0 means 1 leaf node; depth 1 means 1 internal node + 2 leaf nodes. Must be in range [0, 30].'): 2,
   Param(parent='RandomForestRegressor_ea56a97ce9bf', name='numTrees', doc='Number of trees to train (>= 1).'): 100},
  np.float64(286.7714750274078)),
 ({Param(parent='RandomForestRegressor_ea56a97ce9bf', name='maxDepth', doc='Maximum depth of the tree. (>= 0) E.g., depth 0 means 1 leaf node; depth 1 means 1 internal node + 2 leaf nodes. Must be in range [0, 30].'): 4,
   Param(parent='RandomForestRegre

In [20]:
cv = CrossValidator(estimator=rf, # rf
  evaluator=evaluator,
  estimatorParamMaps=paramGrid,
  numFolds=3,
  parallelism=4,
  seed=42)

pipeline = Pipeline(stages=[stringIndexer, vecAssembler, cv]) # cv: put the cross-validator inside the pipeline
pipelineModel = pipeline.fit(trainDF)

26/04/22 03:58:34 WARN DAGScheduler: Broadcasting large task binary with size 1324.0 KiB
26/04/22 03:58:37 WARN DAGScheduler: Broadcasting large task binary with size 1160.7 KiB
26/04/22 03:58:40 WARN DAGScheduler: Broadcasting large task binary with size 1198.4 KiB
26/04/22 03:58:43 WARN DAGScheduler: Broadcasting large task binary with size 1225.2 KiB


In [21]:
predDF = pipelineModel.transform(testDF)
predDF.select("features", "price", "prediction").show(5)

+--------------------+-----+------------------+
|            features|price|        prediction|
+--------------------+-----+------------------+
|(33,[1,3,4,7,8,9,...| 85.0| 143.8612635543596|
|[0.0,2.0,0.0,15.0...| 45.0| 90.56831409179672|
|[0.0,2.0,0.0,15.0...| 70.0|  91.0161235570984|
|(33,[1,3,5,7,8,9,...|128.0| 93.10621215807369|
|(33,[1,3,4,5,7,8,...|159.0|113.12925185885474|
+--------------------+-----+------------------+
only showing top 5 rows



# Cleanup

In [22]:
spark.stop()